# 07 — R-squared metric on controlled cases

This notebook exercises the real `FittingMetricsCalculator` (`pipelines.param_pipeline.metrics`). It builds a synthetic tomogram from known parameters, then evaluates the per-pixel R-squared map for fits of increasing degradation (exact, mildly perturbed, and badly wrong), confirming the metric ranks them correctly.

Modules exercised:

- `pipelines.param_pipeline.metrics.FittingMetricsCalculator` (real, imported and run)
- `tools.data.gaussians.GaussianMixture.evaluate_slice` (real, imported)
- `tools.monitoring.logger.Logger` (real, imported)

Synthetic tomogram written to a temp file, fixed seed, CPU only.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import matplotlib.pyplot as plt

SEED = 0
rng  = np.random.default_rng(SEED)
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
except Exception:
    torch = None

plt.rcParams.update({
    "figure.dpi"     : 110,
    "savefig.dpi"    : 110,
    "font.size"      : 11,
    "axes.grid"      : True,
    "grid.alpha"     : 0.30,
    "grid.linewidth" : 0.4,
})

print("Repo root :", REPO_ROOT)
print("NumPy     :", np.__version__)


In [ ]:
def gaussian_mixture_profile(height_axis, components, noise_std=0.0, rng=None):
    profile = np.zeros_like(height_axis, dtype=np.float64)
    for amp, mu, sigma in components:
        profile += amp * np.exp(-((height_axis - mu) ** 2) / (2.0 * sigma * sigma))
    if noise_std > 0.0:
        draw    = rng.normal(0.0, noise_std, size=height_axis.shape) if rng is not None else np.random.normal(0.0, noise_std, size=height_axis.shape)
        profile = profile + draw
    return profile.astype(np.float32)


def pack_parameters(components_per_pixel, k_max, shape):
    Az, R  = shape
    params = np.zeros((3 * k_max, Az, R), dtype=np.float32)
    for (az, rg), comps in components_per_pixel.items():
        for k, (amp, mu, sigma) in enumerate(comps[:k_max]):
            params[3 * k    , az, rg] = amp
            params[3 * k + 1, az, rg] = mu
            params[3 * k + 2, az, rg] = sigma
    return params


In [ ]:
import tempfile
from pathlib import Path

from tools.data.gaussians import GaussianMixture
from pipelines.param_pipeline.metrics import FittingMetricsCalculator
from tools.monitoring.logger import Logger

logger = Logger(log_dir='', name='nb07_metrics')
print('metrics calculator imported')

## Build a synthetic tomogram from known parameters

The parameter stack has shape `(3*K, Az, R)`. Each pixel carries two Gaussians with randomised but bounded amplitudes, centroids, and spreads. The tomogram of shape `(H, Az, R)` is reconstructed slice-by-slice with the real `evaluate_slice` and lightly corrupted with noise.

In [ ]:
K            = 2
Az, R        = 16, 16
H            = 80
height_range = (0.0, 40.0)
height_axis  = np.linspace(*height_range, H).astype(np.float32)

true_params = np.zeros((3 * K, Az, R), dtype=np.float32)
true_params[0] = rng.uniform(0.6, 1.0,  (Az, R))
true_params[1] = rng.uniform(8.0, 14.0, (Az, R))
true_params[2] = rng.uniform(2.0, 3.5,  (Az, R))
true_params[3] = rng.uniform(0.3, 0.7,  (Az, R))
true_params[4] = rng.uniform(24.0, 32.0,(Az, R))
true_params[5] = rng.uniform(2.5, 4.0,  (Az, R))

tomogram = np.zeros((H, Az, R), dtype=np.float32)
for j, h in enumerate(height_axis):
    tomogram[j] = GaussianMixture.evaluate_slice(true_params, float(h), K)
tomogram += rng.normal(0.0, 0.01, tomogram.shape).astype(np.float32)

tmp_dir  = Path(tempfile.mkdtemp())
tom_path = tmp_dir / 'synthetic_tomogram.npy'
np.save(tom_path, tomogram)
metadata = {'height_range': list(height_range)}
print('tomogram shape:', tomogram.shape, '->', tom_path)

## Three fit qualities

We feed the metrics calculator the exact parameters, a mildly perturbed copy, and a badly wrong copy (centroids and spreads shifted). The R-squared mean should decrease monotonically across the three.

In [ ]:
calc = FittingMetricsCalculator(n_gaussians=K, logger=logger)

perturbed = true_params.copy()
perturbed[2] *= 1.15
perturbed[5] *= 1.15
perturbed[1] += 1.0

wrong = true_params.copy()
wrong[1] += 8.0
wrong[4] -= 8.0
wrong[2] *= 2.0
wrong[5] *= 0.4

cases   = {'exact': true_params, 'perturbed': perturbed, 'wrong': wrong}
metrics = {name: calc.run(p, metadata, tom_path) for name, p in cases.items()}

for name in cases:
    s = metrics[name]['global_summary']
    print(f"{name:10s}  R2 mean={s['r2_mean']:.4f}  median={s['r2_median']:.4f}  neg_frac={s['r2_neg_frac']:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, name in zip(axes, ('exact', 'perturbed', 'wrong')):
    r2 = metrics[name]['r2_map']
    im = ax.imshow(r2, cmap='RdYlGn', vmin=-1.0, vmax=1.0, aspect='auto')
    ax.set_title(f"{name}  (mean R2={metrics[name]['global_summary']['r2_mean']:.3f})")
    ax.set_xlabel('range [px]')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
axes[0].set_ylabel('azimuth [px]')
fig.suptitle('Per-pixel R-squared map across fit qualities')
fig.tight_layout()
plt.show()

## Expected outcome

The exact case should give an R-squared map near 1 everywhere (deep green). The perturbed case should drop modestly, and the wrong case should fall far below, with the mean R-squared ordering exact > perturbed > wrong. This confirms the real metric responds correctly to fit degradation.